In [1]:
import pandas   as  pd
from openpyxl import load_workbook
import sys
import os

In [2]:
# Add the directory containing utils.py to the module search path
utils_path = os.path.abspath('../streamlit/src/')
sys.path.append(utils_path)

# Import the function from utils.py
from utils import analyze_merge

In [3]:
umsteiger_csv = pd.read_csv('/Users/leonardhaas/code/streamlit/data/raw_data/Umsteigeschluessel-KLDB2020-ISCO08.csv',sep=';',header=4)

In [4]:
umsteiger_csv['Bezeichnungen der ISCO-08 (4-Steller)'] = umsteiger_csv['ISCO-08\n(4-Steller)'].fillna('0').astype(int)

In [5]:
# TODO hwo much data do we lose when we ignore the undeutige umsteiger
umsteiger_csv['Umstieg eindeutig (1);\nnicht eindeutig (0)'].value_counts()

Umstieg eindeutig (1);\nnicht eindeutig (0)
1.0    1136
0.0     387
Name: count, dtype: int64

## eindeutig vs uneindeutig

In [6]:
umsteiger_csv.rename(
    columns={'Umstieg eindeutig (1);\nnicht eindeutig (0)': 'umstieg_eindeutig'},
    inplace=True
)

In [7]:
eindeutig = umsteiger_csv.query("umstieg_eindeutig==1")

id_realtion=eindeutig[['KldB 2010\n(5-Steller)','ISCO-08\n(4-Steller)']][:1521]
id_realtion.tail()
id_realtion.columns = ['kldb_2010_key','isco_08_key']

In [8]:
id_realtion['kldb_2010_key'] =id_realtion['kldb_2010_key'].fillna(0).astype(int)

In [9]:
column_dtypes = {

    'bezeichnung_kldb': 'string',    # For text data
    'median_brutto': 'int64',      # For numeric data (int64)
    'average_brutto': 'int64',     # For numeric data (int64)
    'code_kldb': 'int64'             # For integer data
}

In [10]:
verdienst_data_destatis= pd.read_csv('/Users/leonardhaas/code/streamlit/data/raw_data/verdienst_destatis_kldb_5steller.csv')#,dtype=column_dtypes)
verdienst_data_destatis.drop(columns=['Unnamed: 0'], inplace=True)


In [11]:
for column, dtype in column_dtypes.items():
    if dtype == 'int64':
        # Fill NaN with a default value (e.g., 0) before converting
        verdienst_data_destatis[column] = verdienst_data_destatis[column].fillna(0).astype(dtype)
    else:
        # For other types like string, direct conversion works
        verdienst_data_destatis[column] = verdienst_data_destatis[column].astype(dtype)

In [12]:
verdienst_data_destatis = verdienst_data_destatis[(verdienst_data_destatis['median_brutto'] != 0) | (verdienst_data_destatis['average_brutto'] != 0)]

In [13]:
id_verdienst_data_destatis,merge_report=analyze_merge(verdienst_data_destatis,id_realtion,'code_kldb','kldb_2010_key')

Merge Analysis Report:
Total rows in left dataset: 1105
Unique keys in left dataset: 1105
Matched unique keys: 957
Unmatched unique keys: 148
Total rows in merged dataset: 1105
Match Percentage (unique keys): 86.61%

Merge Counts:
both: 957
left_only: 148
right_only: 0


In [14]:
fraktion_daten = pd.read_csv('../streamlit/data/processed_data/fraktion_daten.csv')

In [15]:
fraktion_daten

,Unnamed: 0,ISCO-Code,Berufsgattung(ISCO-Stufe 4),Anzahl,percent,fraktion,major_group
0,1,110,Offiziere in regulären Streitkräften,22030,0.053675,Staatsangestellte,Streitkräfte
1,2,210,Unteroffiziere in regulären Streitkräften,22020,0.053650,NaN,Streitkräfte
2,3,310,Angehörige der regulären Streitkräfte in sonst...,107160,0.261089,Staatsangestellte,Streitkräfte
3,4,1111,Angehörige gesetzgebender Körperschaften,9740,0.023731,Top Management,Führungskräfte
4,5,1112,Leitende Verwaltungsbedienstete,15660,0.038155,Top Management,Führungskräfte
...,...,...,...,...,...,...,...
415,416,9613,Straßenkehrer und verwandte Berufe,6760,0.016470,Dienstleistungsarbeiter,Hilfsarbeitskräfte
416,417,9621,"Boten, Paketauslieferer und Gepäckträger",157600,0.383983,Dienstleistungsarbeiter,Hilfsarbeitskräfte
417,418,9622,Gelegenheitsarbeiter,690,0.001681,Dienstleistungsarbeiter,Hilfsarbeitskräfte
418,419,9623,"Zählerableser, Automatenbefüller und -kassierer",9010,0.021952,Dienstleistungsarbeiter,Hilfsarbeitskräfte


In [16]:
fraktion_verdienst_data, report = analyze_merge(id_verdienst_data_destatis,fraktion_daten,'isco_08_key','ISCO-Code')

Merge Analysis Report:
Total rows in left dataset: 1105
Unique keys in left dataset: 338
Matched unique keys: 338
Unmatched unique keys: 0
Total rows in merged dataset: 1105
Match Percentage (unique keys): 100.00%

Merge Counts:
both: 957
left_only: 148
right_only: 0


In [17]:
print(fraktion_verdienst_data.query('isco_08_key==3119.0')['median_brutto'].mean())
fraktion_verdienst_data.query('isco_08_key==3119.0')['median_brutto'].median()

4518.419354838709


np.float64(4364.0)

## problem kind of duplicates when calculating average

In [18]:
fraktion_verdienst_data['median_brutto_group_mean'] = fraktion_verdienst_data.groupby('isco_08_key')['median_brutto'].transform('mean')

result = fraktion_verdienst_data.groupby([
    'isco_08_key',
    'fraktion',
    'Berufsgattung(ISCO-Stufe 4)',
    'Anzahl'
]).agg({
    'median_brutto_group_mean': 'first'
}).reset_index()


In [50]:
result.to_csv('../streamlit/data/processed_data/isco_verdienst.csv')